In [ ]:
class LLM:
    def generate(self, prompt: str) -> str:
        raise NotImplementedError

In [ ]:
# OpenAI example
from openai import OpenAI
client = OpenAI()

def call_openai(prompt):
    return client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user", "content":prompt}],
        temperature=0.2
    ).choices[0].message.content

In [ ]:
# Local model
from transformers import AutoModelForCausalLM, Tokenizer

tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    device_map="auto",
    load_in_4bit=True
)

def call_local(prompt):
    inputs  = tok(prompt, return_tensors="'pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=256)
    return tok.decode(out[0], skip_special_tokens=True)


In [ ]:
q = rewrite_query(query)
docs = retrieve(q)
docs = rerank(q, docs)

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs):
    pairs = [[query, d] for d in docs]
    scores = reranker.predict(pairs)
    return [d for _, d in sorted(zip(scores, docs), reverse=True)]


In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="docs",
    vectors_config={"size":768, "distance":"Cosine"}
)

client.upsert(
    "docs",
    points=[(i, emb, {"text": doc}) for i,  (emb, doc)in enumerate(data)]
)

In [ ]:
""" 
-RAG reduces hallucinations better than fine-tuning
-Dense retrieval optimizes recall, reranking optimizes precision
-we evaluate retrieval separately from generation
-LoRA gives 90% of fine-tuning performance at 10% cost.
-Agents are workflows not magic
"""